[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/signal38/signal38.github.io/blob/main/notebooks/05_export_gguf.ipynb)

# Notebook 05 — GGUF Export & HuggingFace Hub

Merges the LoRA adapter into LFM2-350M and exports to GGUF (Q4_K_M) for server-side inference via llama.cpp / lm-arena.

**Prerequisites:** Run `02_finetune.ipynb` first to publish the LoRA adapter.

**Requirements:** T4 GPU, `HF_TOKEN` Colab secret (Write access).

**Outputs:** `signal38/lfm2-nk-risk-GGUF` on HuggingFace Hub — Q4_K_M GGUF (~230 MB).

| Cell | What it does | Time |
|------|-------------|------|
| 1 | Install dependencies (restarts runtime once) | ~2 min |
| 2 | Clone repo, verify adapter exists | ~5 s |
| 3 | HuggingFace login | ~5 s |
| 4 | Merge LoRA adapter → fp16 PyTorch model | ~1 min |
| 5 | Export to GGUF (Q4_K_M) and push to Hub | ~2 min |


In [ ]:
import subprocess, sys, os
from pathlib import Path

REPO = Path('/content/signal38.github.io')
if not REPO.exists():
    env = {**os.environ, 'GIT_LFS_SKIP_SMUDGE': '1'}
    subprocess.run(['git', 'clone', '--depth=1', 'https://github.com/signal38/signal38.github.io.git', str(REPO)], check=True, env=env)
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))

from scripts.colab_utils import ensure_notebook_requirements
ensure_notebook_requirements('05_export_gguf', requirements_path=str(REPO / 'requirements.txt'))
# Runtime may restart after this cell.


In [ ]:
import subprocess, sys, os, json
from pathlib import Path

REPO = Path('/content/signal38.github.io')
if not REPO.exists():
    env = {**os.environ, 'GIT_LFS_SKIP_SMUDGE': '1'}
    subprocess.run(['git', 'clone', '--depth=1', 'https://github.com/signal38/signal38.github.io.git', str(REPO)], check=True, env=env)
if str(REPO) not in sys.path:
    sys.path.insert(0, str(REPO))

from scripts.colab_utils import prepare_notebook, require_local_adapter
REPO, PATHS = prepare_notebook(REPO, pull_latest=True)

require_local_adapter(REPO)  # raises if 02_finetune has not run
print('Prerequisites satisfied.')


In [ ]:
from google.colab import userdata
from huggingface_hub import HfApi, login

hf_token = userdata.get('HF_TOKEN')
if not hf_token:
    raise RuntimeError(
        'Missing HF_TOKEN Colab secret.\n'
        'Get a Write-access token at https://huggingface.co/settings/tokens\n'
        'then add it: key icon (left sidebar) → Secrets → Add new secret → name: HF_TOKEN'
    )

login(token=hf_token)
api = HfApi()
print('Logged in.')


In [ ]:
import torch
from unsloth import FastLanguageModel

MAX_SEQ_LENGTH = 2048

# Load in fp16 so the merge produces clean weights without 4bit dequantization noise.
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=str(PATHS['adapter_dir']),
    max_seq_length=MAX_SEQ_LENGTH,
    dtype=torch.float16,
    load_in_4bit=False,
)

merged_dir = PATHS['models_dir'] / 'lfm2-nk-risk' / 'merged'
merged_dir.mkdir(parents=True, exist_ok=True)
model.save_pretrained_merged(str(merged_dir), tokenizer, save_method='merged_16bit')
print(f'Merged model saved: {list(merged_dir.iterdir())}')


In [ ]:
# Export to GGUF (Q4_K_M) and push to Hub.
# Unsloth handles conversion + upload in one call (~2 min, ~230 MB output).
# Note: the uploaded filename is set by unsloth — check the Hub repo after
# this cell and update hf_file in lm-arena config/models.py if it differs.

GGUF_REPO_ID = 'signal38/lfm2-nk-risk-GGUF'
api.create_repo(repo_id=GGUF_REPO_ID, private=False, exist_ok=True)
print(f'Hub GGUF repo ready: https://huggingface.co/{GGUF_REPO_ID}')

model.push_to_hub_gguf(
    GGUF_REPO_ID,
    tokenizer,
    quantization_method='q4_k_m',
    token=hf_token,
)
print(f'Pushed GGUF to https://huggingface.co/{GGUF_REPO_ID}')
